# Modul 2 — Python pentru analiza datelor de mediu

**DELTA-Hub Workshop · 22 mai 2026**

Parcurgem în 75 de minute tot ce e necesar pentru a lucra cu date climatice și geodate în Python:
tipuri de date, formate de fișiere (GeoJSON, GPKG, NetCDF, GeoTIFF, Zarr), calcule statistice,
subsampare, resamplare și vizualizare.

> 📌 **Salvează o copie în Drive** (`File → Save a copy in Drive`) înainte să modifici ceva.

In [ ]:
# ── Setup: rulează o singură dată, ~2 min ─────────────────────────────
!pip install -q numpy pandas geopandas xarray rioxarray zarr netCDF4 matplotlib shapely
import warnings
warnings.filterwarnings('ignore')
print('✓ Pachete instalate.')

---
## 1. Python rapid: tipuri, structuri, funcții

Primele 10 minute — vocabularul de bază Python.

In [ ]:
# ── 1a. Tipuri de date ────────────────────────────────────────────────
x    = 3.14          # float
n    = 365           # int
text = 'Delta Dunării'  # str
flag = True          # bool

print(type(x), x)
print(type(n), n)

# Liste: ordonate, mutabile, elemente mixte
statii = ['Sulina', 'Sf. Gheorghe', 'Crișan', 'Maliuc']
print(statii[0])        # primul element (indexare de la 0)
print(statii[-1])       # ultimul element
print(statii[1:3])      # slice [1, 3)

# Dicționare: chei → valori
coordonate = {
    'Sulina':       (45.15, 29.65),
    'Sf. Gheorghe': (44.90, 29.60),
    'Crișan':       (45.02, 29.42),
}
lat, lon = coordonate['Sulina']
print(f'Sulina: {lat}°N, {lon}°E')

In [ ]:
# ── 1b. Bucle, condiții, funcții ──────────────────────────────────────
for statie, (lat, lon) in coordonate.items():
    print(f'  {statie:20s} → {lat}°N, {lon}°E')

# List comprehension (idiom Python)
latitudini = [coord[0] for coord in coordonate.values()]
print('\nLatitudini:', latitudini)

# Funcție cu docstring
def distanta_euclid(lat1, lon1, lat2, lon2):
    """Distanță aproximativă în grade (nu reală — doar exemplu)."""
    return ((lat2 - lat1)**2 + (lon2 - lon1)**2) ** 0.5

d = distanta_euclid(45.15, 29.65, 44.90, 29.60)
print(f'\nDistanță Sulina → Sf. Gheorghe: {d:.3f}°')

---
## 2. NumPy + pandas

NumPy = calcul numeric vectorizat. pandas = tabele cu etichete.

In [ ]:
import numpy as np

# ── 2a. NumPy: array N-dimensional ───────────────────────────────────
# Temperatura lunară medie la Sulina (valori realiste)
t_vals = np.array([2.1, 4.8, 9.3, 15.6, 21.2, 25.4, 27.8, 27.1, 22.3, 15.1, 8.7, 3.9])
luni   = np.arange(1, 13)

print('Shape:', t_vals.shape)
print('Dtype:', t_vals.dtype)
print(f'Min: {t_vals.min():.1f}°C  Max: {t_vals.max():.1f}°C  Medie: {t_vals.mean():.1f}°C')

# Operații element-wise — fără for loop
t_kelvin = t_vals + 273.15
anomalie = t_vals - t_vals.mean()
print('\nAnomalie față de medie:', np.round(anomalie, 2))

# Indexare logică (masking)
luni_calde = luni[t_vals > 20]
print('Luni cu T > 20°C:', luni_calde)

# Array 2D: grilă lat × lon
grila = np.random.normal(15, 5, (4, 8))   # 4 latitudini × 8 longitudini
print(f'\nGrilă 4×8: {grila.shape}')
print('Medie per latitudine:', grila.mean(axis=1).round(2))

In [ ]:
import pandas as pd

# ── 2b. pandas: DataFrame cu serii temporale ─────────────────────────
idx = pd.date_range('2024-01-01', periods=12, freq='MS')
df  = pd.DataFrame({
    'luna':         idx.month_name(),
    't_medie':      t_vals,
    'precipitatii': np.random.uniform(20, 80, 12).round(1),
}, index=idx)

print(df)
print('\nStatistici:')
print(df[['t_medie', 'precipitatii']].describe().round(2))

# Filtrare
print('\nLuni calde (T > 20°C):')
print(df[df['t_medie'] > 20][['luna', 't_medie']])

# Resamplare trimestrială
print('\nMedie trimestrială:')
print(df['t_medie'].resample('QS').mean().round(2))

---
## 3. Formate de fișiere

Cele 5 formate pe care le vei întâlni cel mai des în analiza datelor de mediu.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# ── 3a. GeoJSON și GPKG (vector) ─────────────────────────────────────
gdf = gpd.GeoDataFrame({
    'name': ['Sulina', 'Sf. Gheorghe', 'Crișan', 'Maliuc', 'Chilia Veche'],
    'type': ['port', 'sat', 'sat', 'rezervatie', 'sat'],
    'pop':  [3200, 800, 250, 120, 1100],
    'geometry': [
        Point(29.65, 45.15),
        Point(29.60, 44.90),
        Point(29.42, 45.02),
        Point(29.38, 45.12),
        Point(29.28, 45.42),
    ]
}, crs='EPSG:4326')

print(gdf)
print(f'\nCRS: {gdf.crs}')
print(f'Extent (bbox): {gdf.total_bounds}')

gdf.to_file('statii_delta.geojson', driver='GeoJSON')
gdf.to_file('statii_delta.gpkg',    driver='GPKG')
print('\n✓ Salvat ca GeoJSON și GPKG')

gdf2 = gpd.read_file('statii_delta.gpkg')
print('\nCitit din GPKG:')
print(gdf2[['name', 'pop']])

In [ ]:
import xarray as xr
import pandas as pd

# ── 3b. NetCDF — creare date sintetice (ERA5-like) ────────────────────
np.random.seed(42)

times = pd.date_range('2024-01-01', '2024-12-31', freq='D')
lats  = np.arange(44.5, 45.75, 0.25)   # 5 puncte
lons  = np.arange(28.5, 30.75, 0.25)   # 9 puncte

# Ciclu sezonier sinusoidal + zgomot gaussian
doy   = np.arange(len(times))
t_base = 15 + 15 * np.sin((doy - 90) * 2 * np.pi / 365)
t2m = (t_base[:, None, None]
       + np.random.normal(0, 2, (len(times), len(lats), len(lons)))
       ).astype('float32')

ds = xr.Dataset(
    {'t2m': (['time', 'lat', 'lon'], t2m)},
    coords={'time': times, 'lat': lats, 'lon': lons},
    attrs={'title': 'Temperatura sintetică Delta Dunării 2024', 'source': 'workshop demo'}
)
ds['t2m'].attrs = {'units': 'degC', 'long_name': '2m air temperature',
                   'standard_name': 'air_temperature'}

print(ds)
print(f'\nDimensiuni: {dict(ds.dims)}')
print(f'Variabile:  {list(ds.data_vars)}')

In [ ]:
# ── 3c. NetCDF — salvare și citire ────────────────────────────────────
ds.to_netcdf('temperatura_delta_2024.nc')
print('✓ Salvat: temperatura_delta_2024.nc')

ds2 = xr.open_dataset('temperatura_delta_2024.nc')
print('\nDataset citit:')
print(ds2)

# Selectare după valori
t_pt = ds2['t2m'].sel(lat=45.0, lon=29.5, method='nearest')
print(f'\nTemp la (45.0N, 29.5E) pe 1 iulie: {float(t_pt.sel(time="2024-07-01")):.2f}°C')

# Slice spațial
subset = ds2['t2m'].sel(lat=slice(44.5, 45.0), lon=slice(29.0, 30.0))
print(f'Subset spațial shape: {subset.shape}')

In [ ]:
import rioxarray

# ── 3d. GeoTIFF — creare și salvare (NDWI sintetic) ──────────────────
ny, nx = 200, 200
lons_r = np.linspace(28.5, 30.5, nx)
lats_r = np.linspace(45.5, 44.5, ny)   # nord → sud

ndwi = np.random.uniform(-1, 1, (ny, nx)).astype('float32')
ndwi[60:120, 40:160] = np.random.uniform(0.1, 0.7, (60, 120))  # canale de apă

da = xr.DataArray(
    ndwi[None, :, :],                       # shape (band, y, x)
    dims=['band', 'y', 'x'],
    coords={'band': [1], 'y': lats_r, 'x': lons_r}
)
da = da.rio.write_crs('EPSG:4326')
da.attrs = {'long_name': 'Normalized Difference Water Index', 'units': '-'}
da.rio.to_raster('ndwi_delta.tif')
print('✓ Salvat: ndwi_delta.tif')
print(da)

In [ ]:
# ── 3e. GeoTIFF — citire și inspecție ────────────────────────────────
da2 = rioxarray.open_rasterio('ndwi_delta.tif')

print('Shape:     ', da2.shape)             # (bands, rows, cols)
print('CRS:       ', da2.rio.crs)
print('Rezoluție: ', da2.rio.resolution())
print('Extent:    ', da2.rio.bounds())
print(f'Valori: min={float(da2.min()):.3f}  max={float(da2.max()):.3f}')

ndwi_band = da2.sel(band=1)
n_apa = int((ndwi_band > 0).sum())
print(f'\nPixeli apă (NDWI > 0): {n_apa} din {ndwi_band.size} ({100*n_apa/ndwi_band.size:.1f}%)')

In [ ]:
# ── 3f. Zarr — format cloud-native ───────────────────────────────────
ds.to_zarr('temperatura_delta.zarr', mode='w')
print('✓ Salvat: temperatura_delta.zarr/')

# Citire lazy
ds_z = xr.open_zarr('temperatura_delta.zarr')
print('\nDataset Zarr:')
print(ds_z)

# Citire cu chunk-uri (util pentru fișiere mari + dask)
ds_z_ch = xr.open_zarr('temperatura_delta.zarr', chunks={'time': 30})
print('\nChunk-uri:', dict(ds_z_ch['t2m'].chunks))

# Pe cloud, sintaxa e aceeași — doar schimbi calea:
# ds_cloud = xr.open_zarr('gs://mybucket/temperatura.zarr')

---
## 4. Calcule de bază

Medie, anomalie, medie mobilă — operații frecvente în analiza climatică.

In [ ]:
t = ds['t2m']   # DataArray cu dimensiunile (time, lat, lon)

# ── Statistici globale ────────────────────────────────────────────────
print(f'Medie globală : {float(t.mean()):.2f}°C')
print(f'Std globală   : {float(t.std()):.2f}°C')
print(f'Min / Max     : {float(t.min()):.2f}°C / {float(t.max()):.2f}°C')

# ── Medie spațială → serie temporală 1D ──────────────────────────────
t_spatial = t.mean(dim=['lat', 'lon'])
print(f'\nSerie temporală shape: {t_spatial.shape}  dims: {t_spatial.dims}')

# ── Anomalie față de media anuală ─────────────────────────────────────
climatology = t.mean(dim='time')    # (lat, lon) — media multi-anuală
anomaly     = t - climatology       # xarray face broadcasting automat
print(f'\nAnomalii shape: {anomaly.shape}')
print(f'Anomalie max pozitivă: {float(anomaly.max()):.2f}°C')

# ── Medie mobilă de 30 de zile (smoothing) ───────────────────────────
t_smooth = t_spatial.rolling(time=30, center=True).mean()
print(f'\nMedie mobilă 30z shape: {t_smooth.shape}')

# ── Masking: valori sub 0°C → NaN ────────────────────────────────────
t_warm = t.where(t > 0)   # valorile <= 0 devin NaN
print(f'Valori de sub 0°C mascate: {int((t <= 0).sum())} celule')

---
## 5. Subsampare

Selectare unui subset din date — fără agregare.

In [ ]:
# ── 5a. Subsampare temporală ─────────────────────────────────────────
print(f'Original: {len(ds.time)} pași zilnici')

# Fiecare a 7-a zi (nu medie — selectare de puncte)
ds_7d = ds.isel(time=slice(None, None, 7))
print(f'Fiecare a 7-a zi  : {len(ds_7d.time)} pași')

# Perioadă specifică
ds_summer = ds.sel(time=slice('2024-06-01', '2024-08-31'))
print(f'Vara 2024 (JJA)   : {len(ds_summer.time)} zile')

# Selectare după condiție (zile calde)
t_mean_daily = ds['t2m'].mean(dim=['lat', 'lon'])
days_hot = ds.isel(time=(t_mean_daily > 25).values)
print(f'Zile cu T > 25°C  : {len(days_hot.time)}')

In [ ]:
# ── 5b. Subsampare spațială ──────────────────────────────────────────
print(f'Original grid: lat={len(ds.lat)}  lon={len(ds.lon)}')

# Fiecare al 2-lea punct
ds_sparse = ds.isel(lat=slice(None, None, 2), lon=slice(None, None, 2))
print(f'Pas 2 → lat={len(ds_sparse.lat)}  lon={len(ds_sparse.lon)}')

# Subset geografic
ds_north = ds.sel(lat=slice(45.0, 45.5), lon=slice(29.0, 30.0))
print(f'Subset nordic: {dict(ds_north.dims)}')

# Coarsen: medie spațială pe blocuri 2×2
ds_coarse = ds.coarsen(lat=2, lon=2, boundary='trim').mean()
print(f'Coarsen 2×2: lat={len(ds_coarse.lat)}  lon={len(ds_coarse.lon)}')

---
## 6. Resamplare

Agregare temporală și interpolări spațiale.

In [ ]:
# ── 6a. Resamplare temporală ─────────────────────────────────────────
# Zilnic → lunar (medie)
ds_monthly = ds.resample(time='1ME').mean()
print(f'Lunar (medie): {ds_monthly["t2m"].shape}')  # (12, nlat, nlon)
print('Timestamps lunare:', ds_monthly.time.values)

# Zilnic → lunar (max)
ds_monthly_max = ds.resample(time='1ME').max()
print(f'\nLunar (max): {ds_monthly_max["t2m"].shape}')

# Zilnic → sezonier (DJF=iarnă, MAM=primăvară, JJA=vară, SON=toamnă)
ds_seasonal = ds.resample(time='QS-DEC').mean()
print(f'\nSezonier: {ds_seasonal["t2m"].shape}')
print('Perioade sezoniere:', ds_seasonal.time.values)

# Zilnic → anual
ds_annual = ds.resample(time='1YE').mean()
print(f'\nAnual: {ds_annual["t2m"].shape}')

In [ ]:
# ── 6b. Resamplare spațială / interpolare ────────────────────────────
# Interpolare la un punct specific (Sulina)
t_sulina = ds['t2m'].interp(lat=45.15, lon=29.65, method='linear')
print(f'Serie Sulina shape: {t_sulina.shape}')
print(f'Temp pe 1 mai 2024: {float(t_sulina.sel(time="2024-05-01")):.2f}°C')

# Interpolare pe grilă mai fină (0.25° → 0.1°)
lats_new = np.arange(44.5, 45.75, 0.1)
lons_new = np.arange(28.5, 30.75, 0.1)
ds_fine  = ds.interp(lat=lats_new, lon=lons_new, method='linear')
print(f'\nOriginal: {dict(ds.dims)}')
print(f'Interpol: {dict(ds_fine.dims)}')

# Schimbare CRS cu rioxarray
# da_utm = da2.rio.reproject('EPSG:32635')   # UTM zona 35N
# print(f'Raster UTM: {da_utm.shape}  CRS: {da_utm.rio.crs}')

---
## 7. Vizualizare

Grafice rapide cu `matplotlib` și `.plot()` nativ xarray.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ── 7a. Serie temporală ───────────────────────────────────────────────
t_mean  = ds['t2m'].mean(dim=['lat', 'lon'])
t_sm    = t_mean.rolling(time=14, center=True).mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_mean.time, t_mean.values, color='#6FB3D8', alpha=0.4,
        linewidth=0.8, label='Zilnic')
ax.plot(t_sm.time,   t_sm.values,   color='#1F4D3F',
        linewidth=2,  label='Medie mobilă 14 zile')
ax.axhline(float(t_mean.mean()), color='#E85D2F', linestyle='--',
           linewidth=1, label=f'Medie anuală: {float(t_mean.mean()):.1f}°C')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.set_ylabel('Temperatură (°C)')
ax.set_title('Temperatura medie spațială — Delta Dunării 2024')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7b. Hartă 2D (medie iulie) ────────────────────────────────────────
t_july = ds['t2m'].sel(time=ds.time.dt.month == 7).mean(dim='time')

fig, ax = plt.subplots(figsize=(8, 5))
t_july.plot(ax=ax, cmap='RdYlBu_r', cbar_kwargs={'label': '°C'})
ax.plot(29.65, 45.15, 'k^', markersize=9, label='Sulina', zorder=5)
ax.plot(29.60, 44.90, 'ks', markersize=7, label='Sf. Gheorghe', zorder=5)
ax.set_title('Temperatura medie — iulie 2024')
ax.set_xlabel('Longitudine')
ax.set_ylabel('Latitudine')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 7c. NDWI raster + stații vector ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raster: NDWI
ndwi_b = da2.sel(band=1)
im = axes[0].imshow(ndwi_b.values,
                    extent=[float(da2.x.min()), float(da2.x.max()),
                            float(da2.y.min()), float(da2.y.max())],
                    cmap='RdYlGn', vmin=-1, vmax=1, origin='upper',
                    aspect='auto')
plt.colorbar(im, ax=axes[0], label='NDWI')
axes[0].set_title('NDWI sintetic — Delta Dunării')
axes[0].set_xlabel('Longitudine')
axes[0].set_ylabel('Latitudine')

# Vector: stații, dimensionate după populație
gdf.plot(ax=axes[1], column='pop', cmap='Oranges',
         markersize=80, legend=True,
         legend_kwds={'label': 'Populație'})
for _, row in gdf.iterrows():
    axes[1].annotate(row['name'],
                     xy=(row.geometry.x, row.geometry.y),
                     xytext=(3, 3), textcoords='offset points', fontsize=8)
axes[1].set_title('Stații de monitorizare — Delta Dunării')
axes[1].set_xlabel('Longitudine')
axes[1].set_ylabel('Latitudine')

plt.tight_layout()
plt.show()

---
## Exercițiu — 10 min

Pornind de la dataset-ul `ds` (temperatura zilnică 2024):

1. **Extrage** seria de timp pentru Sulina (45.15°N, 29.65°E) prin interpolare bilineară.
2. **Calculează** media lunară (`.resample(time='1ME').mean()`).
3. **Găsește** luna cea mai caldă și cea mai rece (`.idxmax()`, `.idxmin()`).
4. **Salvează** seria lunară ca fișier NetCDF și ca CSV.

**Hint:**
```python
t_sulina = ds['t2m'].interp(lat=45.15, lon=29.65, method='linear')
t_monthly = t_sulina.resample(time='1ME').mean()
# → .to_netcdf('sulina_lunar.nc')
# → t_monthly.to_dataframe('t2m').to_csv('sulina_lunar.csv')
```